Goal: Rebuild legacy R code to get the WMC dashboard metrics using the .csv files generated from main

In [38]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import time

In [22]:
# Will try to keep this the same for easier automation in the future
year = "25"
month = "10"

# set working directories - can change in future, be careful
root_dir = "K:/AP/TTM/"
busstate_dir = os.path.join(root_dir, "Data/WMC Dashboard/BusState Cleaned") # Where cleaned busstate data is stored
stops_dir = os.path.join(root_dir, "Data/WMC Dashboard/stops") # where a copy of previous stops data is stored
# NOTE: stops data WILL need to be updated with new med center stops

# pull in necessary static files NOTE: again these WILL need to be updated with new med center stops - may break things downstream so be careful after changing
stop_inventory = pd.read_csv(os.path.join(stops_dir, "stop_inventory.csv"))
pattern_stops = pd.read_csv(os.path.join(stops_dir, "pattern_stops.csv"), header=None)

# have to clean up year/month for proper file reading
# simple map for month number to month abbreviation for file reading
months = {"01": "JAN","02": "FEB","03": "MAR","04": "APR","05": "MAY","06": "JUN","07": "JUL","08": "AUG","09": "SEP","10": "OCT","11": "NOV","12": "DEC"}

year_full = 2000 + int(year)
month_full = months[month]

# pull in cleaned busstate data for month and year of interest
busstate = pd.read_csv(os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv")))

In [23]:
# set directory to save dashbaord data
dashboard_dir = os.path.join(root_dir, f"Data/WMC Dashboard/Dashboard Data/{year_full}/{month_full}")
Path(dashboard_dir).mkdir(parents=True, exist_ok=True)

In [48]:
# original R code selected the 1st and 10th cols
pattern_stops_filtered = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
# name schema per original R code
pattern_stops_filtered.columns = ["ROUTE", "STOP_ID"]

# left join stop inventory 
stops_df = pattern_stops_filtered.merge(stop_inventory, on="STOP_ID", how="left")

In [ ]:
# stops.info()
# sum(stops['ROUTE'] == "MC")

# print(stops[stops['ROUTE'] == "MC"])

<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ROUTE           73 non-null     str    
 1   STOP_ID         73 non-null     int64  
 2   STOP_NAME       72 non-null     str    
 3   TIMEPOINT_NAME  38 non-null     str    
 4   LONG            72 non-null     float64
 5   LAT             72 non-null     float64
 6   HEADING         72 non-null     float64
dtypes: float64(3), int64(1), str(3)
memory usage: 4.1 KB
   ROUTE  STOP_ID            STOP_NAME TIMEPOINT_NAME       LONG        LAT  \
59    MC      403            CARMACK 2          CMCK2 -83.037180  40.001046   
60    MC      404            CARMACK 3          CMCK3 -83.040177  40.000878   
61    MC      405    JOHN HERRICK LOOP           THUB -83.018119  39.997569   
62    MC       27  Outbound to Carmack       OUTBMC86 -83.021652  39.997636   
63    MC       94   CARMACK 5 + STOP 1            N

In [49]:
# create distance function
# NOTE: This does NOT account for curvature of the earth, but locations are close enough that it is negligible
# NOTE: If we want to be more precise in the future, this function can be updated.
def distance(x1, x2, y1, y2):
    '''
    Calculate the distance between two points.

    Args:
        x1 (float): The x-coordinate of the first point longitude.
        x2 (float): The x-coordinate of the second point longitude.
        y1 (float): The y-coordinate of the first point latitude.
        y2 (float): The y-coordinate of the second point latitude.
    Returns:
        float: The distance between the two points.
    '''
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [78]:
# Create function to determine the closest stop
def which_stop(lat, long,  route = "MC", stops = stops_df):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: This will potentially break when new stops are added start Dec. 2025 - be careful when updating stop inventory.

    Args:
        lat (float): The latitude of the point of interest.
        long (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame.
        stops (DataFrame): A DataFrame containing stop information, including 'STOP_ID', 'LAT', and 'LONG' columns.

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # isolate the specified route stops
    selected_stops = stops[stops['ROUTE'] == route].copy()

    # Calculate distance to each stop
    selected_stops['DISTANCE'] = distance(long, selected_stops['LONG'], lat, selected_stops['LAT'])

    # subsetting only stops within 0.0005 units
    selected_stops = selected_stops[selected_stops['DISTANCE'] < 0.0005] # do not know exact units, but this was the threshold used in original R code
    selected_stops = selected_stops.sort_values('DISTANCE')

    # sorted by distance, therfore return first stop which is closest
    if not selected_stops.empty:
        return selected_stops.iloc[0]["STOP_ID"]
    else:
        return None


In [51]:
# which_stop(39.997570, -83.018117)

busstate.info()
busstate.head()

#which_stop(40, -80)

<class 'pandas.DataFrame'>
RangeIndex: 1089062 entries, 0 to 1089061
Data columns (total 23 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   DATE                    1089062 non-null  str    
 1   BUS_ID                  1089062 non-null  int64  
 2   RUN_ID                  1083282 non-null  float64
 3   DEST_SIGN_ROUTE_TEXT    1089046 non-null  str    
 4   BLOCK_ID                1087777 non-null  float64
 5   TRIP_ID                 1080866 non-null  float64
 6   ROUTE_ID                1084654 non-null  str    
 7   STOP_SEQUENCE           1089062 non-null  int64  
 8   LATITUDE                1089062 non-null  float64
 9   LONGITUDE               1089062 non-null  float64
 10  HEADING                 1089062 non-null  int64  
 11  OPERATOR_ID             1072166 non-null  float64
 12  ODOMETER_DISTANCE       1089062 non-null  int64  
 13  TIMEPOINT_ID            121084 non-null   str    
 14  EVENT_TYPE   

,DATE,BUS_ID,RUN_ID,DEST_SIGN_ROUTE_TEXT,BLOCK_ID,TRIP_ID,ROUTE_ID,STOP_SEQUENCE,LATITUDE,LONGITUDE,...,TIMEPOINT_ID,EVENT_TYPE,EVENT_TIME,BOARDINGS,ALIGHTINGS,PASSENGER_LOAD,TRIP_START_TIME,DEPARTURE_TIME,ENTER_STOP_WINDOW_TIME,EXIT_STOP_WINDOW_TIME
0,2025-10-01,1303,1503.0,MC,23675102.0,3733020.0,MC05,3,40.002903,-83.041008,...,NaN,9,00:00:00,0,0,0,23:55:43,NaN,NaN,NaN
1,2025-10-01,1303,1503.0,MC,23675102.0,3733020.0,MC05,3,40.002907,-83.041039,...,NaN,10,00:00:01,0,0,0,23:55:43,NaN,NaN,NaN
2,2025-10-01,1906,2201.0,SS,23674802.0,2300020.0,SS02,2,40.016163,-83.029213,...,NaN,2,00:00:01,0,0,0,23:54:47,NaN,NaN,NaN
3,2025-10-01,1502,1502.0,MC,23675202.0,22020.0,MC05,4,39.999454,-83.028168,...,NaN,2,00:00:03,0,0,0,23:52:34,NaN,NaN,NaN
4,2025-10-01,1501,1501.0,MC,23675302.0,167020.0,MC02,1,39.997444,-83.018684,...,NaN,2,00:00:05,0,0,0,23:57:02,NaN,NaN,NaN


In [ ]:
# Function to process the busstate data for the medical center
def MC_busstate_processing(busstate):
    '''
    Process the busstate data for the medical center route.
    NOTE: This will ONLY work for the medical center route.
    NOTE: This will return a significantly smaller dataframe
    Args:
        busstate (DataFrame): A DataFrame containing busstate data from the cleaned busstate files.
    Returns:
        DataFrame: A pandas DataFrame containing the processed busstate data for the medical center route.
    '''
    processed = (
        busstate

        # filter for MC runs - within 1500 and 1600
        .loc[(busstate['RUN_ID'] >= 1500) & (busstate['RUN_ID'] < 1600)]

        # convert time
        .assign(
            EVENT_TIME = pd.to_datetime(busstate['EVENT_TIME'], format="%H:%M:%S"),
            DEPARTURE_TIME = pd.to_datetime(busstate['DEPARTURE_TIME'], format="%H:%M:%S"),
            ENTER_STOP_WINDOW_TIME = pd.to_datetime(busstate['ENTER_STOP_WINDOW_TIME'], format="%H:%M:%S"),
            EXIT_STOP_WINDOW_TIME = pd.to_datetime(busstate['EXIT_STOP_WINDOW_TIME'], format="%H:%M:%S")
        )

        # arrange by date and even time
        .sort_values(['DATE', 'EVENT_TIME'])

        # Assign stop ID
        # NOTE: Might need to edit this function to address not being at a stop?
        .assign(
            STOP=lambda df: df.apply(lambda row: which_stop(row['LATITUDE'], row['LONGITUDE'], stops=stops_df), axis=1)
        )

        # filter out rows where stop was not assigned
        .loc[lambda df: df['STOP'].notna()] 

        # convert STOP to numeric
        .assign(STOP = lambda df: pd.to_numeric(df['STOP'], errors='coerce'))
        
        # join with stop inventory
        .merge(stop_inventory, how='left', left_on='STOP', right_on='STOP_ID')
        
        # remove dummy stops - R code has 27 and 461 as inbound and outbound dummy stops.
        .loc[lambda df: ~df['STOP'].isin([27, 461])]
        
        # sort by BUS_ID, DATE, EVENT_TIME for elapsed calculations
        .sort_values(['BUS_ID', 'DATE', 'EVENT_TIME'])
        
        # convert times to minutes
        .assign(
            EVENT_TIME_MIN = lambda df: df['EVENT_TIME'].dt.hour * 60 + df['EVENT_TIME'].dt.minute + df['EVENT_TIME'].dt.second / 60,
            DEPARTURE_TIME_MIN = lambda df: df['DEPARTURE_TIME'].dt.hour * 60 + df['DEPARTURE_TIME'].dt.minute + df['DEPARTURE_TIME'].dt.second / 60,
            ENTER_STOP_WINDOW_TIME_MIN = lambda df: df['ENTER_STOP_WINDOW_TIME'].dt.hour * 60 + df['ENTER_STOP_WINDOW_TIME'].dt.minute + df['ENTER_STOP_WINDOW_TIME'].dt.second / 60,
            EXIT_STOP_WINDOW_TIME_MIN = lambda df: df['EXIT_STOP_WINDOW_TIME'].dt.hour * 60 + df['EXIT_STOP_WINDOW_TIME'].dt.minute + df['EXIT_STOP_WINDOW_TIME'].dt.second / 60
        )
        
        # compute elapsed time between rows
        .assign(ELAPSED = lambda df: (df['EVENT_TIME_MIN'] - df['EVENT_TIME_MIN'].shift(1)).round(2))
    )

    # Group and count logic - previously fixed R code - keep as redundant for now, but may be able to simplify in the future
    processed = processed.assign(
        NEW_GROUP = lambda df: (
            df['ELAPSED'].isna() |
            df['STOP'].isna() |
            df['STOP'].shift(1).isna() |
            df['BUS_ID'].isna() |
            df['BUS_ID'].shift(1).isna() |
            (df['STOP'] != df['STOP'].shift(1)) |
            (df['BUS_ID'] != df['BUS_ID'].shift(1)) |
            (df['ELAPSED'] >= 60)
        ).astype(int),
        COUNT = lambda df: df['NEW_GROUP'].cumsum()
    )

    # no longer needed
    processed = processed.drop(columns=['NEW_GROUP'])

    # get rid of non existing even times
    processed = processed.loc[processed['EVENT_TIME'].notna()]

    return processed

In [ ]:
# print(which_stop(-83.02856312, 40.01301304))

None


In [79]:
MC_busstate = MC_busstate_processing(busstate)

In [83]:
MC_busstate.info()
MC_busstate.tail(5)

<class 'pandas.DataFrame'>
Index: 134552 entries, 6801 to 142375
Data columns (total 36 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   DATE                        134552 non-null  str           
 1   BUS_ID                      134552 non-null  int64         
 2   RUN_ID                      134552 non-null  float64       
 3   DEST_SIGN_ROUTE_TEXT        134552 non-null  str           
 4   BLOCK_ID                    134552 non-null  float64       
 5   TRIP_ID                     134213 non-null  float64       
 6   ROUTE_ID                    134298 non-null  str           
 7   STOP_SEQUENCE               134552 non-null  int64         
 8   LATITUDE                    134552 non-null  float64       
 9   LONGITUDE                   134552 non-null  float64       
 10  HEADING_x                   134552 non-null  int64         
 11  OPERATOR_ID                 132478 non-null  float64

,DATE,BUS_ID,RUN_ID,DEST_SIGN_ROUTE_TEXT,BLOCK_ID,TRIP_ID,ROUTE_ID,STOP_SEQUENCE,LATITUDE,LONGITUDE,...,TIMEPOINT_NAME,LONG,LAT,HEADING_y,EVENT_TIME_MIN,DEPARTURE_TIME_MIN,ENTER_STOP_WINDOW_TIME_MIN,EXIT_STOP_WINDOW_TIME_MIN,ELAPSED,COUNT
142365,2025-10-30,2503,1507.0,MC,23675602.0,1136020.0,MC10,0,39.997570,-83.018120,...,THUB,-83.018119,39.997569,360,1201.766667,NaN,NaN,NaN,0.00,33601
142366,2025-10-30,2503,1507.0,MC,23675602.0,1136020.0,MC10,0,39.997570,-83.018120,...,THUB,-83.018119,39.997569,360,1201.766667,NaN,NaN,NaN,0.00,33601
142373,2025-10-30,2503,1507.0,MC,23675602.0,1136020.0,MC10,1,39.997597,-83.018204,...,THUB,-83.018119,39.997569,360,1204.300000,1204.3,1201.533333,1201.75,2.53,33601
142374,2025-10-30,2503,1507.0,MC,23675602.0,1136020.0,MC10,1,39.997597,-83.018204,...,THUB,-83.018119,39.997569,360,1204.300000,NaN,NaN,NaN,0.00,33601
142375,2025-10-30,2503,1507.0,MC,23675602.0,1136020.0,MC10,1,39.997597,-83.018204,...,THUB,-83.018119,39.997569,360,1204.300000,NaN,NaN,NaN,0.00,33601
